# Introduction au Deep Learning avec PyTorch

Dans ce notebook, vous allez découvrir [PyTorch](http://pytorch.org/), un framework pour construire et entraîner des réseaux de neurones.  
PyTorch se comporte, à bien des égards, comme les tableaux que vous connaissez de **Numpy**.  
Après tout, ces tableaux Numpy ne sont rien d’autre que des tenseurs.  

PyTorch prend ces tenseurs et facilite leur transfert vers les **GPU**, afin d’accélérer les calculs nécessaires à l’entraînement des réseaux de neurones.  
Il fournit également un module qui calcule automatiquement les gradients (pour la rétropropagation !) et un autre module dédié à la construction de réseaux de neurones.  

Dans l’ensemble, PyTorch s’intègre de manière plus fluide avec Python et l’écosystème Numpy/Scipy que TensorFlow et d’autres frameworks.


## Réseaux de neurones

Le Deep Learning repose sur les **réseaux de neurones artificiels**, qui existent sous différentes formes depuis la fin des années 1950.  
Les réseaux sont construits à partir d’unités qui imitent les neurones, appelées *unités* ou simplement *neurones*.  

Chaque neurone possède un certain nombre d’entrées pondérées. Ces entrées pondérées sont additionnées (combinaison linéaire), puis passées à travers une **fonction d’activation** pour produire la sortie de l’unité.  

<img src="assets/simple_neuron.png" width=400px>

Mathématiquement, cela s’écrit :

$$
y = f(w_1 x_1 + w_2 x_2 + b) \\\\
y = f\left(\sum_i w_i x_i + b \right)
$$

En notation vectorielle, c’est le produit scalaire (produit interne) de deux vecteurs :

$$
h = \begin{bmatrix}
x_1 \, x_2 \cdots  x_n
\end{bmatrix}
\cdot 
\begin{bmatrix}
           w_1 \\\\
           w_2 \\\\
           \vdots \\\\
           w_n
\end{bmatrix}
$$


## Tenseurs

En réalité, les calculs dans les réseaux de neurones ne sont qu’une série d’opérations d’algèbre linéaire sur des **tenseurs**, une généralisation des matrices.  

- Un vecteur est un tenseur à 1 dimension.  
- Une matrice est un tenseur à 2 dimensions.  
- Un tableau avec trois indices est un tenseur à 3 dimensions (par exemple une image en couleur RGB).  

La structure de données fondamentale pour les réseaux de neurones est donc le **tenseur**, et PyTorch (comme la plupart des frameworks de deep learning) est construit autour de ce concept.  

<img src="assets/tensor_examples.svg" width=600px>

Avec ces bases, il est temps d’explorer comment utiliser PyTorch pour construire un réseau de neurones simple.


In [1]:
import torch

In [2]:
def activation(x):
    """ Sigmoid activation function 
    
        Arguments
        ---------
        x: torch.Tensor
    """
    return 1/(1+torch.exp(-x))

In [3]:
### Generate some data
torch.manual_seed(7) # Set the random seed so things are predictable

# Features are 3 random normal variables
features = torch.randn((1, 5))
# True weights for our data, random normal variables again
weights = torch.randn_like(features)
# and a true bias term
bias = torch.randn((1, 1))

In [4]:
print("features : ", features)
print("weights : ", weights)
print("biais : ", bias)

print("features dimensions : ", features.shape)
print("weights dimensions : ", weights.shape)
print("bias dimensions : ", bias.shape)


features :  tensor([[-0.1468,  0.7861,  0.9468, -1.1143,  1.6908]])
weights :  tensor([[-0.8948, -0.3556,  1.2324,  0.1382, -1.6822]])
biais :  tensor([[0.3177]])
features dimensions :  torch.Size([1, 5])
weights dimensions :  torch.Size([1, 5])
bias dimensions :  torch.Size([1, 1])


Ci-dessus, j’ai généré des données que nous pouvons utiliser pour calculer la sortie de notre petit réseau.  
Pour l’instant, ce ne sont que des valeurs aléatoires, mais par la suite nous utiliserons de vraies données.  

Explication de chaque ligne :

- `features = torch.randn((1, 5))` crée un tenseur de taille `(1, 5)` (une ligne, cinq colonnes) contenant des valeurs tirées d’une loi normale centrée réduite (moyenne = 0, écart-type = 1).  
- `weights = torch.randn_like(features)` crée un autre tenseur de la même taille que `features`, également rempli de valeurs aléatoires selon une loi normale.  
- `bias = torch.randn((1, 1))` génère une seule valeur issue d’une loi normale.  

Les tenseurs PyTorch peuvent être additionnés, multipliés, soustraits, etc., comme des tableaux Numpy.  
En général, vous utiliserez les tenseurs PyTorch comme des tableaux Numpy, avec en plus certains avantages tels que l’accélération GPU (que nous verrons plus tard).  

👉 **Exercice** : Calculez la sortie du réseau avec les `features`, `weights` et `bias`.  
Comme en Numpy, PyTorch propose la fonction [`torch.sum()`](https://pytorch.org/docs/stable/torch.html#torch.sum) ou la méthode `.sum()` pour effectuer des sommes.  
Utilisez la fonction `activation` définie ci-dessus comme fonction d’activation.


In [5]:
print("shape after view : " , weights.view(-1, 1).shape)

shape after view :  torch.Size([5, 1])


In [6]:
## Calculate the output of this network using the weights and bias tensors
output = torch.mm(features, weights.view(-1, 1)) + bias
output

tensor([[-1.6619]])

In [7]:
combinaison_lineaire = torch.sum(features * weights) + bias
combinaison_lineaire

tensor([[-1.6619]])

In [8]:
y = activation(combinaison_lineaire)
y

tensor([[0.1595]])

Vous pouvez effectuer la multiplication et l’addition en une seule opération grâce à une **multiplication matricielle**. En général, vous utiliserez les multiplications matricielles car elles sont plus efficaces et accélérées par les bibliothèques modernes et le calcul haute performance sur GPU.

Ici, nous voulons faire une multiplication matricielle entre les **features** et les **weights**. Pour cela, nous pouvons utiliser [`torch.mm()`](https://pytorch.org/docs/stable/torch.html#torch.mm) ou [`torch.matmul()`](https://pytorch.org/docs/stable/torch.html#torch.matmul), ce dernier étant un peu plus complexe car il prend en charge le *broadcasting*. Si nous essayons directement avec `features` et `weights` tels quels, nous obtiendrons une erreur :

```python
>> torch.mm(features, weights)

---------------------------------------------------------------------------
RuntimeError                              Traceback (most recent call last)
<ipython-input-13-15d592eb5279> in <module>()
----> 1 torch.mm(features, weights)

RuntimeError: size mismatch, m1: [1 x 5], m2: [1 x 5] at /Users/soumith/minicondabuild3/conda-bld/pytorch_1524590658547/work/aten/src/TH/generic/THTensorMath.c:2033

Lorsque vous construirez des réseaux de neurones dans n’importe quel framework, vous verrez souvent cette erreur. Ce qui se passe ici, c’est que nos tenseurs n’ont pas les bonnes dimensions pour réaliser une multiplication matricielle.
Rappel : pour une multiplication de matrices, le nombre de colonnes de la première doit être égal au nombre de lignes de la seconde. Or, features et weights ont tous deux la forme (1, 5). Il faut donc changer la forme de weights pour que la multiplication matricielle fonctionne.

Astuce : pour voir la forme d’un tenseur tensor, utilisez tensor.shape. Si vous construisez des réseaux, vous utiliserez cette méthode très souvent.

Plusieurs options existent : weights.reshape()
, weights.resize_()
, et weights.view()
.

weights.reshape(a, b) : renvoie un nouveau tenseur avec les mêmes données que weights de taille (a, b) (parfois une vue, parfois une copie).

weights.resize_(a, b) : modifie en place le tenseur avec la nouvelle forme. Si la nouvelle forme a moins d’éléments, certains sont supprimés (mais pas en mémoire) ; si elle en a plus, les nouveaux éléments sont non initialisés. L’underscore _ indique une opération in-place. Voir cette discussion utile sur les opérations in-place
.

weights.view(a, b) : renvoie une vue du tenseur avec la taille (a, b).

J’utilise généralement .view(), mais les trois méthodes fonctionnent. Nous pouvons donc remodeler weights en une matrice de 5 lignes et 1 colonne avec : weights.view(5, 1).

Exercice : Calculez la sortie de notre petit réseau en utilisant la multiplication matricielle

In [9]:
## Calculate the output of this network using matrix multiplication

### Empilons-les !

C’est ainsi que vous pouvez calculer la sortie d’un seul neurone.  
La véritable puissance de cet algorithme se manifeste lorsque vous commencez à **empiler ces unités individuelles** en couches, puis en ensembles de couches, formant ainsi un réseau de neurones.  

La sortie d’une couche devient l’entrée de la couche suivante. Avec plusieurs unités d’entrée et plusieurs unités de sortie, nous devons maintenant exprimer les poids sous forme de matrice.

<img src='assets/multilayer_diagram_weights.png' width=450px>

La première couche (en bas) correspond aux **entrées**, appelée la **couche d’entrée**.  
La couche du milieu est appelée la **couche cachée**, et la dernière couche (à droite) est la **couche de sortie**.  

Nous pouvons exprimer ce réseau mathématiquement à l’aide de matrices et utiliser la multiplication matricielle pour obtenir les combinaisons linéaires de chaque unité en une seule opération.  

Par exemple, la couche cachée ($h_1$ et $h_2$ ici) peut être calculée comme suit :

$$
\vec{h} = [h_1 \, h_2] = 
\begin{bmatrix}
x_1 \, x_2 \cdots \, x_n
\end{bmatrix}
\cdot 
\begin{bmatrix}
           w_{11} & w_{12} \\\\
           w_{21} & w_{22} \\\\
           \vdots & \vdots \\\\
           w_{n1} & w_{n2}
\end{bmatrix}
$$

La sortie de ce petit réseau est obtenue en considérant la couche cachée comme les entrées de l’unité de sortie.  
Le résultat du réseau s’exprime simplement par :

$$
y =  f_2 \! \left(\, f_1 \! \left(\vec{x} \, \mathbf{W_1}\right) \mathbf{W_2} \right)
$$


In [10]:
### Generate some data
torch.manual_seed(7) # Set the random seed so things are predictable

# Features are 3 random normal variables
features = torch.randn((1, 3))

# Define the size of each layer in our network
n_input = features.shape[1]     # Number of input units, must match number of input features
n_hidden = 2                    # Number of hidden units 
n_output = 1                    # Number of output units

# Weights for inputs to hidden layer
W1 = torch.randn(n_input, n_hidden)
# Weights for hidden layer to output layer
W2 = torch.randn(n_hidden, n_output)

# and bias terms for hidden and output layers
B1 = torch.randn((1, n_hidden))
B2 = torch.randn((1, n_output))

> **Exercice :** calculez la sortie de ce réseau multicouche avec les poids `W1` et `W2`, ainsi que les biais `B1` et `B2`. 

In [11]:
torch.mm(features, W1)

tensor([[ 0.6270, -0.3969]])

In [12]:
## Your solution here
out1 = torch.sum(torch.mm(features, W1))+B1
out2 = torch.sum(torch.mm(out1,W2))+B2
activation(out2)

tensor([[0.4371]])

Si vous avez fait cela correctement, vous devriez voir la sortie suivante :  
`tensor([[ 0.3171]])`.

Le nombre d’unités cachées est un paramètre du réseau, souvent appelé **hyperparamètre** pour le distinguer des paramètres que sont les poids et les biais.  
Comme vous le verrez plus tard lorsque nous parlerons de l’entraînement d’un réseau de neurones, plus un réseau possède d’unités cachées et de couches, mieux il sera capable **d’apprendre à partir des données** et de produire des prédictions précises.


## De Numpy à Torch et retour

Section bonus !  
PyTorch offre une fonctionnalité très pratique pour convertir entre les tableaux **Numpy** et les tenseurs **Torch**.  

- Pour créer un tenseur à partir d’un tableau Numpy, utilisez :  
  `torch.from_numpy()`

- Pour convertir un tenseur en tableau Numpy, utilisez la méthode :  
  `.numpy()`


In [13]:
import numpy as np
np.set_printoptions(precision=8)
a = np.random.rand(4,3)
a

array([[0.20036324, 0.62726579, 0.42053344],
       [0.89692254, 0.11339254, 0.78855465],
       [0.3200287 , 0.40485684, 0.13634451],
       [0.27008199, 0.21302519, 0.81459066]])

In [14]:
torch.set_printoptions(precision=8)
b = torch.from_numpy(a)
b

tensor([[0.20036324, 0.62726579, 0.42053344],
        [0.89692254, 0.11339254, 0.78855465],
        [0.32002870, 0.40485684, 0.13634451],
        [0.27008199, 0.21302519, 0.81459066]], dtype=torch.float64)

In [15]:
b.numpy()

array([[0.20036324, 0.62726579, 0.42053344],
       [0.89692254, 0.11339254, 0.78855465],
       [0.3200287 , 0.40485684, 0.13634451],
       [0.27008199, 0.21302519, 0.81459066]])

La mémoire est partagée entre le tableau Numpy et le tenseur Torch.  
Ainsi, si vous modifiez les valeurs **in-place** (directement) dans l’un des objets, l’autre sera également modifié.


In [16]:
# Multiply PyTorch Tensor by 2, in place
b.mul_(2)

tensor([[0.40072649, 1.25453158, 0.84106687],
        [1.79384507, 0.22678508, 1.57710930],
        [0.64005740, 0.80971368, 0.27268903],
        [0.54016399, 0.42605038, 1.62918131]], dtype=torch.float64)

In [17]:
# Numpy array matches new values from Tensor
a

array([[0.40072649, 1.25453158, 0.84106687],
       [1.79384507, 0.22678508, 1.5771093 ],
       [0.6400574 , 0.80971368, 0.27268903],
       [0.54016399, 0.42605038, 1.62918131]])